In [ ]:
import os
import json
from glob import glob
from tqdm import tqdm
import numpy as np
from PIL import Image, ImageDraw

# 경로 설정
data_root = '../../data/precise_BC_cell_scoring/er_pr/'
image_dir = os.path.join(data_root, 'patch_images')
label_dir = os.path.join(data_root, 'labels')
review_dir = os.path.join(data_root, 'review_overlay')
os.makedirs(review_dir, exist_ok=True)

# 클래스별 색상
CLASS_COLORS = {
    0: (0, 255, 0),      # class0 - green
    1: (255, 255, 0),    # class1 - yellow
    2: (255, 165, 0),    # class2 - orange
    3: (255, 0, 0),      # class3 - red
    4: (128, 128, 128),  # other  - gray
}
POINT_RADIUS = 3

In [ ]:
# 모든 ER/PR 패치에 대해 라벨 중점 포인트를 클래스 색상으로 오버레이한 검수 이미지 생성
json_list = sorted(glob(os.path.join(label_dir, '*.json')))
print(f"총 {len(json_list)} 개 라벨")

for json_path in tqdm(json_list):
    fname = os.path.splitext(os.path.basename(json_path))[0]
    img_path = os.path.join(image_dir, fname + '.png')
    if not os.path.exists(img_path):
        continue

    out_path = os.path.join(review_dir, fname + '.png')
    if os.path.exists(out_path):
        continue  # 이미 생성된 검수 이미지는 건너뜀 (수동 삭제 보존)

    with open(json_path, 'r') as f:
        labels = json.load(f)

    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    draw = ImageDraw.Draw(img)

    for lab in labels:
        cls_id = int(lab['class_id'])
        cx = float(lab['cx']) * W
        cy = float(lab['cy']) * H
        color = CLASS_COLORS.get(cls_id, (255, 255, 255))
        draw.ellipse(
            [cx - POINT_RADIUS, cy - POINT_RADIUS,
             cx + POINT_RADIUS, cy + POINT_RADIUS],
            fill=color, outline=color,
        )

    img.save(out_path)

print(f"검수용 오버레이 이미지 저장 완료 -> {review_dir}")
print("이제 review_overlay 폴더를 직접 열어 잘못된 패치의 오버레이 이미지를 삭제하세요.")

In [ ]:
# class 1/2/3 (positive) 라벨인데 bbox 내부에 DAB 갈색 발현이 없는 "의심 라벨"을 포함한 패치만 저장
# class 3 처럼 발현이 매우 강해 거의 검게 보이는 경우도 positive 로 인정 (dark-DAB 보정)
suspect_dir = os.path.join(data_root, 'review_suspect')
os.makedirs(suspect_dir, exist_ok=True)

POSITIVE_CLASSES = {1, 2, 3}

# 일반 갈색 (중/약 DAB) : HSV 기준
BROWN_H_LOW, BROWN_H_HIGH = 8, 35
BROWN_S_MIN = 50
BROWN_V_MAX = 220

# 진한 DAB (거의 검은색으로 보임) : 어둡고, hematoxylin 의 파란/보라와 구분하기 위해 R >= B
DARK_V_MAX = 90
DARK_RB_DIFF = 5        # R - B >= 5 이면 파란 계열이 아님 (갈색/검정 DAB)

# bbox 내부에서 "positive 픽셀" 비율이 이 값 미만이면 의심으로 판정
POS_PIXEL_RATIO = 0.05

def positive_mask(np_rgb):
    hsv = np.array(Image.fromarray(np_rgb).convert('HSV'))
    H_ch, S_ch, V_ch = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    R = np_rgb[..., 0].astype(np.int16)
    B = np_rgb[..., 2].astype(np.int16)

    light_brown = (H_ch >= BROWN_H_LOW) & (H_ch <= BROWN_H_HIGH) & \
                  (S_ch >= BROWN_S_MIN) & (V_ch <= BROWN_V_MAX)
    dark_dab = (V_ch <= DARK_V_MAX) & ((R - B) >= DARK_RB_DIFF)
    return light_brown | dark_dab

def has_positive(np_rgb_crop):
    if np_rgb_crop.size == 0:
        return False
    return positive_mask(np_rgb_crop).mean() >= POS_PIXEL_RATIO

json_list = sorted(glob(os.path.join(label_dir, '*.json')))



In [ ]:
# positive 조건을 만족하지 못하는 class 1/2/3 라벨을 class 0 으로 재지정하여 JSON 덮어쓰기
# (바로 위 셀의 positive_mask / has_positive 함수를 재사용)
json_list = sorted(glob(os.path.join(label_dir, '*.json')))

relabel_total = 0
relabel_per_class = {1: 0, 2: 0, 3: 0}
affected_patches = 0

for json_path in tqdm(json_list):
    fname = os.path.splitext(os.path.basename(json_path))[0]
    img_path = os.path.join(image_dir, fname + '.png')
    if not os.path.exists(img_path):
        continue

    with open(json_path, 'r') as f:
        labels = json.load(f)

    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    np_img = np.array(img)

    changed = False
    for lab in labels:
        cls_id = int(lab['class_id'])
        if cls_id not in POSITIVE_CLASSES:
            continue
        cx = float(lab['cx']) * W
        cy = float(lab['cy']) * H
        bw = float(lab['w']) * W
        bh = float(lab['h']) * H
        x1 = max(0, int(round(cx - bw / 2)))
        y1 = max(0, int(round(cy - bh / 2)))
        x2 = min(W, int(round(cx + bw / 2)))
        y2 = min(H, int(round(cy + bh / 2)))
        if x2 <= x1 or y2 <= y1:
            continue
        crop = np_img[y1:y2, x1:x2]
        if not has_positive(crop):
            lab['class_id'] = 0
            relabel_total += 1
            relabel_per_class[cls_id] += 1
            changed = True

    if changed:
        with open(json_path, 'w') as f:
            json.dump(labels, f)
        affected_patches += 1

print(f"재지정된 라벨 총 개수: {relabel_total}")
print(f"클래스별 재지정 수: {relabel_per_class}")
print(f"영향받은 패치 수: {affected_patches} / 전체 {len(json_list)}")

In [ ]:
# review_overlay 폴더에서 사용자가 삭제한 이미지에 해당하는
# 전처리 데이터(patch_images / labels) 의 짝(image + json) 도 삭제
remaining = {os.path.splitext(os.path.basename(p))[0]
             for p in glob(os.path.join(review_dir, '*.png'))}

label_files = sorted(glob(os.path.join(label_dir, '*.json')))
removed = 0
for json_path in label_files:
    fname = os.path.splitext(os.path.basename(json_path))[0]
    if fname in remaining:
        continue
    img_path = os.path.join(image_dir, fname + '.png')
    if os.path.exists(img_path):
        os.remove(img_path)
    if os.path.exists(json_path):
        os.remove(json_path)
    removed += 1

print(f"삭제된 짝(image + json): {removed} 개")
print(f"남은 라벨 수: {len(glob(os.path.join(label_dir, '*.json')))}")